# Cross-Validation

- Used for Evaluating model performance and selecting the best model.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("melbourne_preprocessed_data.csv")

In [3]:
print(df.shape)
df.head()

(27243, 13)


,Price2,Distance_sqrt,Landsize_KNN,BuildingArea_KNN,Bathroom_mode,Age,CouncilArea_Encoded,Method_Encoded,Lattitude_impute,Longtitude_impute,Type_h,Type_t,Rooms
0,1480000.0,1.581139,202.0,156.834586,1.0,56.0,1.103285e+06,1.051479e+06,-37.7996,144.9984,1.0,0.0,2
1,1035000.0,1.581139,156.0,79.000000,1.0,116.0,1.103285e+06,1.051479e+06,-37.8079,144.9934,1.0,0.0,2
2,1465000.0,1.581139,134.0,150.000000,2.0,117.0,1.103285e+06,8.790954e+05,-37.8093,144.9944,1.0,0.0,3
3,850000.0,1.581139,94.0,156.834586,2.0,57.0,1.103285e+06,1.117884e+06,-37.7969,144.9969,1.0,0.0,3
4,1600000.0,1.581139,120.0,142.000000,1.0,2.0,1.103285e+06,1.194341e+06,-37.8072,144.9941,1.0,0.0,4


In [5]:
X = df.drop("Price2", axis=1)
y = df["Price2"]

In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [34]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1),
    "Lasso Regression": Lasso(alpha=0.001),
    "Elastic Net": ElasticNet(alpha=0.001, l1_ratio=0.5),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "SVR Regression": SVR(),
    "KNN Regression": KNeighborsRegressor(n_neighbors=5)
}

In [27]:
from sklearn.model_selection import cross_val_score

## Leave-One-Out Cross-Validation (LOOCV)

Leave-One-Out Cross-Validation (LOOCV) is a cross-validation technique where one observation is used as the test set and all remaining observations are used for training.

This process is repeated for every observation in the dataset, and the final performance is calculated as the average of all iterations.

### Advantages
- Uses almost all data for training.
- Suitable for small datasets.

### Disadvantages
- Computationally expensive for large datasets.
- Training time increases significantly as the dataset size grows.

### Example
For a dataset with 100 samples:
- Number of folds = 100
- Train on 99 samples
- Test on 1 sample
- Repeat 100 times

In [32]:
# Takes too much time, Computationally Expensive

from sklearn.model_selection import LeaveOneOut

loo = LeaveOneOut() # LOOCV

results = []

for name, model in models.items():

    scores = cross_val_score(model, X_train, y_train, cv=loo, scoring="neg_mean_squared_error")

    results.append({
        "model": name,
        "RMSE" : -scores.mean(),
    })

results_df = pd.DataFrame(results)
results_df.sort_values("RMSE")

KeyboardInterrupt: 

In [40]:
df.shape

(27243, 13)

## K-Fold Cross-Validation

K-Fold Cross-Validation is a model evaluation technique where the dataset is divided into **K equal folds**.

The model is trained on **K−1 folds** and tested on the remaining fold. This process is repeated **K times**, with each fold being used as the test set once.

The final performance is calculated as the average of all K iterations.

### Advantages
- Provides a more reliable performance estimate than a single train-test split.
- Uses all observations for both training and testing.
- Reduces evaluation bias.

### Disadvantages
- More computationally expensive than a train-test split.
- Training time increases as the number of folds increases.

### Example
For a dataset with 100 samples and K = 5:
- Number of folds = 5
- Train on 80 samples
- Test on 20 samples
- Repeat 5 times
- Average the results

### Common Choice
- K = 5 or K = 10 are the most commonly used values.

In [38]:
from sklearn.model_selection import KFold, cross_validate
import pandas as pd

kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = []

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=kf,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error"
        }
    )

    results.append({
        "Model": name,
        "MAE": -scores["test_mae"].mean(),
        "RMSE": -scores["test_rmse"].mean()
    })

results_df = pd.DataFrame(results)
print(results_df)

               Model        MAE       RMSE
0  Linear Regression 251,610.13 389,921.42
1   Ridge Regression 251,599.02 389,921.23
2   Lasso Regression 251,610.13 389,921.42
3        Elastic Net 251,535.76 389,935.84
4      Decision Tree 234,102.57 404,614.84
5      Random Forest 174,388.86 306,030.66
6     SVR Regression 423,753.18 659,487.89
7     KNN Regression 250,106.65 421,373.04


In [39]:
pd.set_option("display.float_format", "{:,.2f}".format)

results_df.sort_values("RMSE")

,Model,MAE,RMSE
5,Random Forest,"174,388.86","306,030.66"
1,Ridge Regression,"251,599.02","389,921.23"
2,Lasso Regression,"251,610.13","389,921.42"
0,Linear Regression,"251,610.13","389,921.42"
3,Elastic Net,"251,535.76","389,935.84"
4,Decision Tree,"234,102.57","404,614.84"
7,KNN Regression,"250,106.65","421,373.04"
6,SVR Regression,"423,753.18","659,487.89"


## Random Forest is best performing model with lowest RMSE and MAE among all the models we have trained.

## Stratified K-Fold Cross-Validation

Stratified K-Fold Cross-Validation is a variation of K-Fold Cross-Validation that preserves the class distribution in each fold.

Each fold contains approximately the same proportion of classes as the original dataset.

### Advantages
- Maintains class balance in every fold.
- Provides a more reliable evaluation for imbalanced datasets.
- Reduces bias caused by uneven class distribution.

### Disadvantages
- Applicable only to classification problems.
- Slightly more computationally expensive than K-Fold.

### Example
For a dataset with:
- Class 0 = 90%
- Class 1 = 10%

Each fold will maintain approximately the same 90:10 class ratio.

### Common Choice
- n_splits = 5 or 10

### Use Case
- Classification problems with imbalanced classes.
- Fraud Detection
- Disease Prediction
- Spam Detection

In [ ]:
# Syntax

from sklearn.model_selection import StratifiedKFold, cross_val_score
import pandas as pd

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y,
        cv=skf,
        scoring="accuracy"
    )

    results.append({
        "Model": name,
        "Accuracy": scores.mean()
    })

results_df = pd.DataFrame(results)
results_df.sort_values("Accuracy", ascending=False)

## Time Series Cross-Validation

Time Series Cross-Validation is a validation technique designed for time-series data. It preserves the chronological order of observations and prevents future data from being used during training.

In each iteration, the training set grows while the test set moves forward in time.

### Advantages
- Preserves temporal order.
- Prevents data leakage.
- Suitable for forecasting problems.

### Disadvantages
- Not suitable for non-time-series data.
- Can be computationally expensive for large datasets.

### Example

Fold 1:
- Train: Jan–Mar
- Test: Apr

Fold 2:
- Train: Jan–Apr
- Test: May

Fold 3:
- Train: Jan–May
- Test: Jun

### Use Cases
- Stock Price Prediction
- Sales Forecasting
- Weather Forecasting
- Energy Demand Prediction

In [42]:
# Syntax

from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.linear_model import LinearRegression

tscv = TimeSeriesSplit(n_splits=5)

model = LinearRegression()

scores = cross_val_score(
    model,
    X,
    y,
    cv=tscv,
    scoring="neg_root_mean_squared_error"
)

print("Average RMSE:", -scores.mean())